# 🎓 Zbuduj własną sieć neuronową!
## 🕹️ Misja: MNIST – Rozpoznawanie ręcznie pisanych cyfr

Twoim zadaniem jest:

1. **Samodzielnie zbudować sieć neuronową** – użyj `nn.Sequential`, `nn.Linear`, funkcji aktywacji...
2. **Samodzielnie przeprowadzić trening** – wybierz liczbę epok, batch size, funkcję kosztu, optymalizator...
3. **Poprawić model** – np. dodaj kolejną warstwę, zmień liczbę neuronów, użyj Dropout albo ReLU zamiast Sigmoid.
4. **Sprawdź dokładność** modelu na danych testowych.

---

## 📌 Twój cel:
Zbuduj model, który osiąga przynajmniej **90% dokładności** na danych testowych.  


In [ ]:
# 📦 Import bibliotek
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from tqdm import tqdm

# 🔧 Ustawienie urządzenia
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Używane urządzenie: {device}")

In [ ]:
# 📥 Wczytanie i przetwarzanie danych MNIST
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

BATCH_SIZE = 64

trainset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True)

testset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=BATCH_SIZE, shuffle=False)


In [ ]:
# 👁️‍🗨️ Podgląd kilku przykładowych danych
examples = enumerate(trainloader)
batch_idx, (example_data, example_targets) = next(examples)

fig = plt.figure()
for i in range(6):
    plt.subplot(2, 3, i + 1)
    plt.tight_layout()
    plt.imshow(example_data[i][0], cmap='gray')
    plt.title(f"Cyfra: {example_targets[i]}")
    plt.xticks([])
    plt.yticks([])
plt.show()


In [ ]:
# 🧠 Definicja sieci neuronowej przy użyciu nn.Sequential

# Trzeba uzupełnić odpowiednio nn.Sequential (nn.Flatten zostaje jako podpowiedź :D)
model = nn.Sequential(
    nn.Flatten(),                # Spłaszczenie obrazu 28x28 -> 784
).to(device)

print(model)


In [ ]:
# ⚙️ Funkcja kosztu i optymalizator
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [ ]:
# 🏋️‍♂️ Trenowanie modelu
epochs = 5

for epoch in tqdm(range(epochs)):
    for batch_idx, (inputs, labels) in enumerate(trainloader):
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()

        # Trzeba zdobyć wynik po przepuszczniu inputu przez model
        # obliczyć loss na podstawie outputów i labelów
        outputs = ...
        loss = ...


        # 🔍 Debug: wypisz stratę co 100 batchy
        if batch_idx % 1000 == 0:
            print(f"[Batch {batch_idx}] Strata: {loss.item():.4f}")

        # Zaktualizuj loss
        # Zaktualizuj optimizer


    print(f"Epoka {epoch+1} zakończona")


In [ ]:
# 📊 Ocena modelu na danych testowych

def test_model(model):

  correct = 0
  total = 0

  with torch.no_grad():
      for inputs, labels in testloader:
          inputs, labels = inputs.to(device), labels.to(device)
          outputs = model(inputs)
          _, predicted = torch.max(outputs.data, 1)
          total += labels.size(0)
          correct += (predicted == labels).sum().item()

  print(f"Dokładność na zbiorze testowym: {100 * correct / total:.2f}%")
  return 100 * correct / total


In [ ]:
test_model(model)

# 🧪 Zadania dodatkowe:
- Zmień liczbę neuronów w warstwie ukrytej
- Dodaj kolejne warstwy
- Zastosuj dropout, batch normalization
- Zmieniaj liczbę epok i ucz się, jak wpływa to na wynik

# 🚀 Zadanie Boss-Fight (⭐): Konwolucyjne Sieci Neuronowe (CNN)

Udało Ci się wyciągnąć ponad 90% na zwykłym MLP? Super! Ale... zwykłe sieci w pełni połączone (Linear) traktują obrazek jak jeden długi, płaski sznurek pikseli. Przez to tracą mnóstwo informacji o tym, co leży obok czego (np. że piksele tworzą poziomą kreskę).

Do obrazów prawdziwi specjaliści używają **Sieci Konwolucyjnych (CNN - Convolutional Neural Networks)**. Zamiast patrzeć na cały obrazek naraz, używają one małych okienek ("filtrów"), które przesuwają się po obrazku i szukają wzorców (krawędzi, kółek, kształtów).

Twoja misja ostateczna: **Zbuduj sieć CNN i przebij barierę 98% dokładności!** Tym razem nie podajemy gotowego kodu. Musisz poszperać w internecie/dokumentacji.

### To-Do List 📝:

- [ ] **Research:** Wygoogluj, jak działają warstwy `nn.Conv2d` oraz `nn.MaxPool2d` (lub `nn.AvgPool2d`) w PyTorch.
- [ ] **Nowa Architektura:** Zbuduj nowy model. Typowa struktura to: `Conv2d` -> `ReLU` -> `MaxPool2d` -> `Conv2d` -> `ReLU` -> `MaxPool2d`.
- [ ] **Spłaszczanie (Flatten):** Zanim przekażesz wynik z warstw konwolucyjnych do warstwy decyzyjnej (`nn.Linear`), musisz spłaszczyć dane (rozprostować je z siatki 2D do wektora). Użyj `nn.Flatten()`.
- [ ] **Wymiary Tensorów (Najtrudniejsze!):** Kiedy przepuścisz obraz 28x28 przez filtry i pooling, jego rozmiar się zmniejszy. Musisz policzyć (albo sprawdzić metodą prób i błędów/printów), ile cech wychodzi z ostatniego poolinga, żeby podać odpowiednią liczbę wejść do pierwszej warstwy `Linear`.
- [ ] **Kształt Wejścia:** Zwróć uwagę, jak podajesz dane z `DataLoader`. MLP wymagało spłaszczenia wejścia. CNN przyjmuje obrazy "w całości", tzn. w formacie `[batch_size, channels, height, width]` (dla MNIST to `[batch_size, 1, 28, 28]`). Zmodyfikuj swoją pętlę treningową, jeśli trzeba!
- [ ] **Trening:** Odpal trening na tych samych danych. Zobaczysz, że CNN uczy się szybciej i osiąga znacznie lepsze wyniki na obrazkach!